<b><font size="6" color="#E8800A">Week 8a · Similarity and KNN</font></b><br>
<b><font size="4">The nearest training rows, and how they are found</font></b><br>

K-nearest neighbours learns nothing about the outcome when it is fitted. It
stores the training athletes, and when asked about a new athlete it finds the stored
athletes nearest to that athlete and answers with their outcomes. The prediction therefore
rests on a distance between rows, on how many neighbours answer, and on how the
nearest ones are found.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>What "nearest" means</font>](#distance)
* [<font color='#E8800A'>Fitting KNN</font>](#fit)
* [<font color='#E8800A'>How KNN predicts</font>](#predict)
* [<font color='#E8800A'>What KNN cannot do</font>](#limits)
* [<font color='#E8800A'>How the neighbours are found</font>](#algorithm)
* [<font color='#E8800A'>The settings of KNN</font>](#knn-params)
* [<font color='#E8800A'>Radius neighbours, the alternative</font>](#radius)
* [<font color='#E8800A'>Nested cross-validation</font>](#nested)
* [<font color='#E8800A'>The grid search</font>](#search)
* [<font color='#E8800A'>The number we stand behind</font>](#closing)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>What "nearest" means</font> <a class="anchor" id="distance"></a>
[Back to TOC](#toc)

A new athlete is answered by the nearest training athletes, so the first question is how far apart two athletes are. Each prepared athlete is a row of numbers, and the distance between two rows is built from their differences, column by column: the steps below compute one by hand, and the general formula follows.

In [ ]:
from tqdm.auto import tqdm

import sys
from pathlib import Path

# course_helpers.py and preprocessing.py sit in this folder, next to the notebook.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Circle, Rectangle
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.neighbors import (
    BallTree,
    KDTree,
    KNeighborsClassifier,
    RadiusNeighborsClassifier,
)

from course_helpers import (
    PLOT_BLUE,
    PLOT_ORANGE,
    Scoreboard,
    record,
    result_context,
    results_before,
)

random_state = 33
np.random.seed(random_state)  # For reproducibility
figsize = (7, 5)

# Fold-safe encoding means a rare level can occur only in validation, so this
# message would repeat once per scored fold.
warnings.filterwarnings(
    "ignore",
    message="Found unknown categories.*during transform",
    category=UserWarning,
)

from preprocessing import (
    CachedSearchCV,
    FoldPreprocessor,
    PreparedEstimator,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
classification_recipe_log = "../../logs/week_04_feature_work_classification_log.json"


__Step 1:__ Load the classification spine and split it.

Run this notebook from its own folder inside the course
repository: that is what makes the `data/` paths below work. If you
downloaded this file on its own from Moodle, move it into the repository before
you run it.

In [ ]:
champions, classification_recipe = load_recipe(
    "../../data/interim/champions.csv",
    classification_recipe_log,
)
# RecordID names each row, so it becomes the index.
champions = champions.set_index("RecordID")
X_churn = champions.loc[:, classification_recipe["categorical"] + classification_recipe["numeric"]].copy()
y_churn = champions["Outcome"]

X_churn_train, X_churn_test, y_churn_train, y_churn_test = train_test_split(
    X_churn, y_churn, test_size=0.2, stratify=y_churn, random_state=random_state,
)
print(f"{len(X_churn_train):,} training athletes, {len(X_churn_test):,} test athletes,"
      f" positive rate {y_churn_train.mean():.4f}")

__Step 2:__ Prepare the athletes, and fix the folds every score is read on.

In [ ]:
churn_prep = FoldPreprocessor(classification_recipe).fit(X_churn_train, y_churn_train)
X_churn_train_ready = churn_prep.transform(X_churn_train)
X_churn_test_ready = churn_prep.transform(X_churn_test)

# The ten folds every validation score is read on.
churn_folds = outer_splitter("classification", random_state=random_state)
churn_prepared = prepared_folds(
    FoldPreprocessor(classification_recipe), churn_folds,
    X_churn_train, y_churn_train, n_jobs=4)

__Step 3:__ Two training athletes, column by column: each one's prepared value, their difference and its square.

In [ ]:
prepared_names = list(churn_prep.get_feature_names_out())
first, second = X_churn_train_ready[0], X_churn_train_ready[1]
by_column = pd.DataFrame({"first athlete": first, "second athlete": second,
                          "difference": first - second,
                          "squared difference": (first - second) ** 2},
                         index=prepared_names)
by_column.round(3)

__Step 4:__ Add the squared differences, and take the square root.

In [ ]:
squared_sum = by_column["squared difference"].sum()
print(f"sum of the squared differences: {squared_sum:.4f}")
print(f"its square root, the distance:  {np.sqrt(squared_sum):.4f}")

The squared differences add up to 11.7071: `Supplements` contributes the largest, 2.837, each of the three one-hot columns in which the two differ exactly 1, and every shared value nothing.

The same computation for any two rows $a$ and $b$ is the **Euclidean distance**

$$d(a, b) = \sqrt{\sum_{j=1}^{m} (a_j - b_j)^2}.$$

Here $m$ is the number of prepared columns, $j$ runs over them, and $a_j$ and $b_j$ are the two rows' values in column $j$. Each column adds the square of one difference, so a difference of 2 in one column adds as much as differences of 1 in four columns.

The **Minkowski distance** replaces the square and the square root by a power $p$ and its root:

$$d_p(a, b) = \Big(\sum_{j=1}^{m} \lvert a_j - b_j \rvert^{p}\Big)^{1/p},$$

where $\lvert a_j - b_j \rvert$ is the absolute difference. With $p = 2$ it is the Euclidean distance above, and with $p = 1$ it is the **Manhattan** distance, the sum of the absolute differences. The larger $p$, the more the largest single difference decides the distance. `KNeighborsClassifier` uses the Minkowski distance with `p=2` unless told otherwise.

<div class="alert alert-block alert-info">

**In raw units, each column is weighted by its variance.** Write $\sigma_j$ for the standard deviation of column $j$ over the training rows, and $z_{aj}$ for row $a$'s standardised value in it: its value minus the column mean, divided by $\sigma_j$. Then

$$\sum_{j} (a_j - b_j)^2 = \sum_{j} \sigma_j^2\,(z_{aj} - z_{bj})^2 .$$

A distance on raw columns is a distance on standardised columns in which column $j$ counts $\sigma_j^2$ times, so a column written in large units outweighs the others whatever it says about the outcome. The prepared columns of Step 2 are already scaled by the inherited recipe.

</div>

# <font color='#E8800A'>Fitting KNN</font> <a class="anchor" id="fit"></a>
[Back to TOC](#toc)

Fitting stores the prepared training athletes and their outcomes. Nothing about the outcome is estimated: there is no coefficient, mean or variance to learn, and every prediction is computed from the stored rows when it is asked for. Depending on `algorithm`, fitting also builds a structure over the stored rows that a search can use.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html'>sklearn.neighbors.KNeighborsClassifier(n_neighbors=5, *, weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='minkowski', metric_params=None, n_jobs=None)</a>

**Definition:**
K-nearest neighbours classification. Stores the training rows and predicts the class held by most of the `n_neighbors` nearest of them.

**Main Parameters:**
- `n_neighbors` (default `5`): how many stored rows answer.
- `weights` (default `'uniform'`): `'uniform'` gives every neighbour one vote; `'distance'` weights each vote by one over the neighbour's distance.
- `algorithm` (default `'auto'`): how the neighbours are found, `'brute'`, `'kd_tree'` or `'ball_tree'`; `'auto'` chooses one from the data.
- `leaf_size` (default `30`): where a tree stops splitting the stored rows.
- `p` (default `2`): the power of the Minkowski distance, `1` Manhattan and `2` Euclidean.
- `metric` (default `'minkowski'`): the distance.

**Main Attributes (after fitting):**
- `classes_`: the class labels, in the order `predict_proba` uses.
- `n_samples_fit_`: how many training rows are stored.

**Main Methods:**
- `fit(X, y)`: stores the rows, and builds a tree if `algorithm` asks for one.
- `kneighbors(X)`: the distances to the nearest stored rows and their positions.
- `predict_proba(X)`: the share of each class among the neighbours.
- `predict(X)`: the class most of the neighbours hold.
</div>

__Step 5:__ Score `KNeighborsClassifier()` on the folds, and open the log.

In [ ]:
knn_model = KNeighborsClassifier()
fold_models = [clone(knn_model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(churn_prepared, desc='churn prepared', leave=True, position=0)]
knn_cv = {
    "train": [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"]))
              for fold, fitted in zip(churn_prepared, fold_models)],
    "validation": [f1_score(fold["y_score"], fitted.predict(fold["X_score"]))
                   for fold, fitted in zip(churn_prepared, fold_models)]}
# The same model with its recipe, fitted on all the training rows: later steps read both parts.
knn_fitted = PreparedEstimator(FoldPreprocessor(classification_recipe), clone(knn_model)).fit(
    X_churn_train, y_churn_train)

In [ ]:
churn_log = []
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": "KNeighborsClassifier()",
                   "train": knn_cv["train"],
                   "validation": knn_cv["validation"], "model": knn_fitted},
       X_churn_test, y_churn_test, metric=f1_score, label="F1")

**F1 0.8315 on the validation folds and 0.8364 on the test rows.** The train F1, 0.8833, sits above both: every training athlete is one of its own five nearest neighbours, at distance zero, so its own outcome is one of the five votes.

# <font color='#E8800A'>How KNN predicts</font> <a class="anchor" id="predict"></a>
[Back to TOC](#toc)

A prediction needs only the stored rows and the distance. The steps below follow the first test athlete to a prediction, one method at a time.

__Step 6:__ `kneighbors`: the distances from the first test athlete to the five nearest training athletes, and their positions among the stored rows.

In [ ]:
knn = knn_fitted.estimator_
newcomer_ready = knn_fitted.preprocessor_.transform(X_churn_test.iloc[[0]])
distances, positions = knn.kneighbors(newcomer_ready)
print(f"distances: {distances[0].round(4)}")
print(f"positions: {positions[0]}")

__Step 7:__ The first of those distances by hand, from the athlete's prepared row and the stored row at the first position.

In [ ]:
nearest = X_churn_train_ready[positions[0, 0]]
print(f"by hand: {np.sqrt(((newcomer_ready[0] - nearest) ** 2).sum()):.4f}")

**0.1845 both ways.** `kneighbors` computed the Euclidean distance of Step 4 and kept the five smallest, in increasing order.

__Step 8:__ The outcomes of the five neighbours, read from the training labels at those positions.

In [ ]:
y_churn_train.iloc[positions[0]]

__Step 9:__ `predict_proba`: the share of winners among the five neighbours.

In [ ]:
print(f"share of winners among the five: {y_churn_train.iloc[positions[0]].mean():.1f}")
print(f"predict_proba:                   {knn.predict_proba(newcomer_ready)[0]}")

__Step 10:__ `predict`: the class most of the five hold, beside the athlete's outcome.

In [ ]:
print(f"predict:               {knn.predict(newcomer_ready)[0]}")
print(f"the athlete's outcome: {y_churn_test.iloc[0]}")

**Four of the five neighbours won, so `predict_proba` gives this athlete 0.8 of winning and `predict` returns 1.** The test athlete won. The probability is the share of the five neighbours in each class, and the prediction is the class with the larger share.

# <font color='#E8800A'>What KNN cannot do</font> <a class="anchor" id="limits"></a>
[Back to TOC](#toc)

Both limits below follow from answering with the stored neighbours, and each is read from a printed number.

__Step 11:__ Count the probabilities of winning the default model gives the test athletes.

In [ ]:
test_probability = knn_fitted.predict_proba(X_churn_test)[:, 1]
pd.Series(test_probability).value_counts().sort_index()

**The 800 test athletes receive only six probabilities, 0, 0.2, 0.4, 0.6, 0.8 and 1, and 227 of them receive exactly 1.** With five neighbours a probability is a count of winners divided by five, so the model sorts the athletes into six groups and no finer.

__Step 12:__ Build a probe athlete unlike any training athlete: the training athlete whose `Train bf competition` is closest to its median, with every numeric column set to ten times its largest training value.

In [ ]:
numeric = classification_recipe["numeric"]
typical_index = X_churn_train["Train bf competition"].sub(
    X_churn_train["Train bf competition"].median()).abs().idxmin()
probe = X_churn_train.loc[[typical_index]].copy()
probe.loc[:, numeric] = 10 * X_churn_train[numeric].max().to_numpy()
probe[numeric]

__Step 13:__ Ask the default model about the probe, then set the probe's distance to its nearest training athlete beside the median of that distance over the test athletes.

In [ ]:
probe_ready = knn_fitted.preprocessor_.transform(probe)
print(f"probe's probability of winning: {knn.predict_proba(probe_ready)[0, 1]:.1f}")

In [ ]:
probe_distance = knn.kneighbors(probe_ready, n_neighbors=1)[0][0, 0]
print(f"its nearest training athlete: {probe_distance:.3f} away")

In [ ]:
test_distance = np.median(knn.kneighbors(
    knn_fitted.preprocessor_.transform(X_churn_test), n_neighbors=1)[0])
print(f"test athletes' nearest, median: {test_distance:.3f} away")

**The probe is 64.665 from its nearest training athlete, against a median of 1.148 for the test athletes, and the model still answers it: a probability of winning of 0.2.** Nothing in the answer says that the neighbours behind it are far away.

# <font color='#E8800A'>How the neighbours are found</font> <a class="anchor" id="algorithm"></a>
[Back to TOC](#toc)

The neighbours `kneighbors` returns do not depend on the method that finds them; the number of distances computed, and the time, do. `algorithm` chooses among three methods.

**Brute force** (`'brute'`) computes the distance to every stored athlete and keeps the five smallest.

**A k-d tree** (`'kd_tree'`, Bentley, 1975), built at fit time, cuts the stored athletes in two at the median of the column with the largest range, its largest value minus its smallest, and cuts each half the same way, so every node is a box. A search skips any box whose nearest edge is farther than the fifth-best distance so far.

**A ball tree** (`'ball_tree'`, Omohundro, 1989) splits the same way, but each node is a ball around the mean of its athletes, reaching the farthest. A search skips a ball when the distance to its centre minus its radius exceeds the fifth-best distance.

Splitting stops at the **leaves**, nodes of `leaf_size` to twice `leaf_size` athletes, where every distance is computed. `'auto'` picks a method from the data.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KDTree.html'>sklearn.neighbors.KDTree(X, leaf_size=40, metric='minkowski', **kwargs), and sklearn.neighbors.BallTree with the same signature</a>

**Definition:**
A k-d tree or a ball tree over the rows of `X`, for finding nearest neighbours.

**Main Parameters:**
- `X`: the rows to store.
- `leaf_size` (default `40`): a leaf holds between `leaf_size` and twice `leaf_size` rows.

**Main Attributes (after fitting):**
- `data`: the stored rows.

**Main Methods:**
- `query(X, k=1)`: the distances to the `k` nearest stored rows and their positions.
- `get_arrays()`: the stored rows, their order, one record per node (`idx_start`, `idx_end`, `is_leaf`, `radius`) and the node bounds: box corners or ball centres.
- `reset_n_calls()` and `get_n_calls()`: set to zero, and read, the count of distances computed.
</div>

__Step 14:__ Take a seeded sample of 40 training athletes on `Train bf competition` and `Strength training`, build both trees on it with `leaf_size=5`, so every node can be drawn, and count their nodes and levels.

In [ ]:
pair_columns = ["Train bf competition", "Strength training"]
pair = X_churn_train_ready[:, [prepared_names.index(name) for name in pair_columns]]
sample = pair[np.random.RandomState(random_state).choice(len(pair), 40, replace=False)]
kd_tree = KDTree(sample, leaf_size=5)
ball_tree = BallTree(sample, leaf_size=5)

In [ ]:
# get_arrays returns the stored rows, the order the tree keeps them in, one record
# per node and the node bounds: box corners for the k-d tree, ball centres for the
# ball tree. Node i holds the rows order[idx_start:idx_end].
_, kd_order, kd_nodes, kd_bounds = kd_tree.get_arrays()
_, ball_order, ball_nodes, ball_centres = ball_tree.get_arrays()
# The number of levels is fixed when a tree is built, from the rows and leaf_size.
levels = int(np.log2(max(1, (len(sample) - 1) / 5)) + 1)
print(f"{len(kd_nodes)} nodes in each tree, {kd_nodes['is_leaf'].sum()} of them leaves;"
      f" {levels} levels")

__Step 15:__ The k-d root's cut by hand: each column's range, its largest value minus its smallest.

In [ ]:
ranges = pd.Series(sample.max(axis=0) - sample.min(axis=0), index=pair_columns)
cut_column = int(np.argmax(ranges))
ranges.round(3)

__Step 16:__ Sort the column with the larger range: the 20 smallest go to node 1 and the rest to node 2, and the cut lies between the 20th and 21st values.

In [ ]:
by_value = np.argsort(sample[:, cut_column])
smaller_half, larger_half = by_value[:20], by_value[20:]
print(f"cut column: {pair_columns[cut_column]}")
print(f"20th smallest value: {sample[by_value[19], cut_column]:.3f}")
print(f"21st smallest value: {sample[by_value[20], cut_column]:.3f}")

__Step 17:__ Check the two halves against nodes 1 and 2 of the fitted tree.

In [ ]:
node_one = kd_order[kd_nodes["idx_start"][1]:kd_nodes["idx_end"][1]]
node_two = kd_order[kd_nodes["idx_start"][2]:kd_nodes["idx_end"][2]]
print(f"node 1 holds the 20 smallest: {set(smaller_half) == set(node_one)}")
print(f"node 2 holds the other 20:    {set(larger_half) == set(node_two)}")

__Step 18:__ The ball root by hand: the mean of the 40 athletes and the distance to the farthest, checked against the fitted tree.

In [ ]:
root_centre = sample.mean(axis=0)
root_radius = np.linalg.norm(sample - root_centre, axis=1).max()
print(f"centre by hand {root_centre.round(3)}, fitted {ball_centres[0, 0].round(3)}")
print(f"radius by hand {root_radius:.3f}, fitted {ball_nodes['radius'][0]:.3f}")

**The root cuts `Train bf competition`, whose range, 4.051, is larger than the 2.515 of `Strength training`: the spread that picks the column is this range, not the variance.** The 20 smallest athletes, up to 0.302, form node 1 and the 20 from 0.338 up form node 2, exactly the fitted tree's rows, because scikit-learn sends the smaller half, `n // 2` rows, to the first child. The ball root is centred on the mean of the 40, with radius 2.939, as fitted.

__Step 19:__ Every node by the same rule: its rows, its cut column and cut values, and its ball's radius.

In [ ]:
node_rows = []
for node in range(len(kd_nodes)):
    rows = sample[kd_order[kd_nodes["idx_start"][node]:kd_nodes["idx_end"][node]]]
    row = {"node": node, "depth": int(np.log2(node + 1)), "rows": len(rows),
           "radius": ball_nodes["radius"][node]}
    if not kd_nodes["is_leaf"][node]:
        # The cut column has the largest range over the node's own rows; the cut lies
        # between the largest value of its first child and the smallest of its second.
        column = int(np.argmax(rows.max(axis=0) - rows.min(axis=0)))
        first, second = [sample[kd_order[kd_nodes["idx_start"][child]:kd_nodes["idx_end"][child]]]
                         for child in (2 * node + 1, 2 * node + 2)]
        row.update({"cut_column": pair_columns[column],
                    "cut_after": first[:, column].max(), "cut_before": second[:, column].min()})
    node_rows.append(row)
tree_nodes = pd.DataFrame(node_rows).set_index("node")
tree_nodes.round(3)

__Step 20:__ Check every ball against the k-d node with its number.

In [ ]:
print("each ball holds the rows of the k-d node with its number:",
      all(set(ball_order[start:end]) == set(kd_order[start:end])
          for start, end in zip(kd_nodes["idx_start"], kd_nodes["idx_end"])))

In [ ]:
balls = [sample[ball_order[start:end]]
         for start, end in zip(ball_nodes["idx_start"], ball_nodes["idx_end"])]
print("each ball is centred on the mean of its rows:",
      all(np.allclose(ball_centres[0, node], rows.mean(axis=0)) for node, rows in enumerate(balls)))

In [ ]:
print("each radius reaches the farthest of its rows:",
      all(np.isclose(ball_nodes["radius"][node],
                     np.linalg.norm(rows - ball_centres[0, node], axis=1).max())
          for node, rows in enumerate(balls)))

**Each child applies the same rule to its own 20 athletes:** node 1 cuts `Train bf competition` between -0.321 and -0.271, and node 2 cuts `Strength training` between 0.324 and 0.408. The three checks print `True`.

__Step 21:__ Place every node in the diagram, with its labels.

In [ ]:
# Node i sits at depth log2(i + 1), and its children are nodes 2i + 1 and 2i + 2.
layout = pd.DataFrame({"x": [(node + 1.5 - 2 ** depth) / 2 ** depth
                             for node, depth in tree_nodes["depth"].items()],
                       "y": -tree_nodes["depth"]}, index=tree_nodes.index)
parents = [(node - 1) // 2 if node else 0 for node in tree_nodes.index]
layout["parent_x"], layout["parent_y"] = layout["x"].loc[parents].to_numpy(), layout["y"].loc[parents].to_numpy()
layout["cut_at"] = (tree_nodes["cut_after"] + tree_nodes["cut_before"]) / 2
layout["kd_label"] = [f"node {node}\n{rows} rows" for node, rows in tree_nodes["rows"].items()]
layout["kd_cut"] = [f"\ncut {column}\nbetween {after:.3f} and {before:.3f}"
                    if isinstance(column, str) else "" for column, after, before
                    in zip(tree_nodes["cut_column"], tree_nodes["cut_after"], tree_nodes["cut_before"])]
layout["ball_label"] = [f"node {node}\n{rows} rows\nradius {radius:.2f}" for node, rows, radius
                        in zip(tree_nodes.index, tree_nodes["rows"], tree_nodes["radius"])]
layout[["x", "y", "parent_x", "parent_y", "cut_at"]].round(3)

In [ ]:
figure, axes = plt.subplots(3, 2, figsize=(13, 14), layout="constrained",
                            gridspec_kw={"width_ratios": [1, 1.15]})
for level, (space, diagram) in enumerate(axes):
    space.scatter(sample[:, 0], sample[:, 1], s=16, color="#4A4A4A", zorder=3)
    for node in tree_nodes.index[tree_nodes["depth"] <= level]:
        newest = tree_nodes.loc[node, "depth"] == level
        # The box is the smallest box around the node's own rows.
        lower, upper = kd_bounds[0, node], kd_bounds[1, node]
        space.add_patch(Rectangle(lower, *(upper - lower), fill=False,
                                  edgecolor=PLOT_BLUE if newest else "lightgrey",
                                  linewidth=2.5 if newest else 1))
        if newest:
            space.text(*(lower + upper) / 2, str(node), ha="center", va="center", fontsize=11,
                       fontweight="bold", color=PLOT_BLUE, zorder=4,
                       bbox=dict(boxstyle="circle", facecolor="white", edgecolor=PLOT_BLUE))
        else:
            # The cut this node made: a dashed line between its two halves' values.
            cut = layout.loc[node, "cut_at"]
            colour = PLOT_BLUE if tree_nodes.loc[node, "depth"] == level - 1 else "lightgrey"
            ends = ([cut, cut], [lower[1], upper[1]])
            if tree_nodes.loc[node, "cut_column"] == pair_columns[1]:
                ends = ([lower[0], upper[0]], [cut, cut])
            space.plot(*ends, color=colour, linestyle="--", linewidth=1.5)
        diagram.plot(layout.loc[node, ["x", "parent_x"]], layout.loc[node, ["y", "parent_y"]],
                     color="grey", linewidth=1, zorder=1)
        diagram.text(layout.loc[node, "x"], layout.loc[node, "y"],
                     layout.loc[node, "kd_label"] + ("" if newest else layout.loc[node, "kd_cut"]),
                     ha="center", va="center", fontsize=9, zorder=2,
                     color="white" if newest else "black",
                     bbox=dict(boxstyle="round", facecolor=PLOT_BLUE if newest else "white",
                               edgecolor=PLOT_BLUE if newest else "grey"))
    space.set(xlabel=f"{pair_columns[0]} (prepared)", ylabel=f"{pair_columns[1]} (prepared)",
              title=f"Depth {level}: the athletes, the boxes and the cuts")
    diagram.set(xlim=(0, 1), ylim=(-2.6, 0.6), title=f"Depth {level}: the tree")
    diagram.axis("off")
figure.suptitle("How the k-d tree is built: each node's rows are cut at the median of the"
                " column with the largest range")
plt.show()

**Each level cuts every box in two.** The solid lines are the boxes, each the smallest box around its node's athletes, and the dashed lines are the cuts, between the two halves' values, so sibling boxes need not touch. Nodes 1 and 2 hold 20 athletes each, and the four leaves, nodes 3 to 6, hold 10 each.

In [ ]:
figure, axes = plt.subplots(3, 2, figsize=(13, 14), layout="constrained",
                            gridspec_kw={"width_ratios": [1, 1.15]})
for level, (space, diagram) in enumerate(axes):
    space.scatter(sample[:, 0], sample[:, 1], s=16, color="#4A4A4A", zorder=3)
    for node in tree_nodes.index[tree_nodes["depth"] <= level]:
        newest = tree_nodes.loc[node, "depth"] == level
        space.add_patch(Circle(ball_centres[0, node], tree_nodes.loc[node, "radius"], fill=False,
                               edgecolor=PLOT_ORANGE if newest else "lightgrey",
                               linewidth=2.5 if newest else 1))
        if newest:
            space.text(*ball_centres[0, node], str(node), ha="center", va="center", fontsize=11,
                       fontweight="bold", color=PLOT_ORANGE, zorder=4,
                       bbox=dict(boxstyle="circle", facecolor="white", edgecolor=PLOT_ORANGE))
        diagram.plot(layout.loc[node, ["x", "parent_x"]], layout.loc[node, ["y", "parent_y"]],
                     color="grey", linewidth=1, zorder=1)
        diagram.text(layout.loc[node, "x"], layout.loc[node, "y"], layout.loc[node, "ball_label"],
                     ha="center", va="center", fontsize=9, zorder=2,
                     color="white" if newest else "black",
                     bbox=dict(boxstyle="round", facecolor=PLOT_ORANGE if newest else "white",
                               edgecolor=PLOT_ORANGE if newest else "grey"))
    space.set_aspect("equal")
    space.set(xlabel=f"{pair_columns[0]} (prepared)", ylabel=f"{pair_columns[1]} (prepared)",
              title=f"Depth {level}: the athletes and the balls")
    diagram.set(xlim=(0, 1), ylim=(-2.6, 0.6), title=f"Depth {level}: the tree")
    diagram.axis("off")
figure.suptitle("How the ball tree is built: the same split, with each node's rows wrapped in a"
                " ball around their mean")
plt.show()

**The same split, but each node is a ball.** The root's ball needs a radius of 2.939 to reach its farthest athlete and covers far more of the plane than the athletes do, and the balls of one level overlap: node 3's, of radius 1.814, reaches into those of nodes 4 and 5.

__Step 22:__ One query, searched three ways on the same 40 athletes: the first test athlete. Count the distances each method computes.

In [ ]:
query = X_churn_test_ready[:1, [prepared_names.index(name) for name in pair_columns]]
computed = {"brute force": len(sample)}
for name, tree in [("k-d tree", kd_tree), ("ball tree", ball_tree)]:
    tree.reset_n_calls()
    distances, _ = tree.query(query, k=5)
    computed[name] = tree.get_n_calls()
reach = distances[0, -1]
print(f"distance to the fifth neighbour: {reach:.4f}")
print(f"distances computed: {computed}")

__Step 23:__ Name, by node number, the leaves that meet the circle through the fifth neighbour, which no search can skip, and count their athletes.

In [ ]:
leaves = np.flatnonzero(kd_nodes["is_leaf"])
# A box meets the circle when the point of the box nearest the query is within reach.
nearest_point = np.clip(query, kd_bounds[0, leaves], kd_bounds[1, leaves])
kd_met = leaves[np.linalg.norm(nearest_point - query, axis=1) <= reach]
# A ball meets it when the distance to its centre, minus its radius, is within reach.
ball_met = leaves[np.linalg.norm(ball_centres[0, leaves] - query, axis=1)
                  - ball_nodes["radius"][leaves] <= reach]
for name, met in [("k-d tree", kd_met), ("ball tree", ball_met)]:
    print(f"{name}: leaves {list(map(int, met))} meet the circle,"
          f" holding {tree_nodes.loc[met, 'rows'].sum()} athletes")

In [ ]:
figure, panels = plt.subplots(1, 3, figsize=(16, 6), sharex=True, sharey=True)
panels[0].scatter(sample[:, 0], sample[:, 1], s=16, color="#4A4A4A", zorder=3)
for leaf in leaves:
    lower, upper = kd_bounds[0, leaf], kd_bounds[1, leaf]
    panels[1].add_patch(Rectangle(lower, *(upper - lower), linewidth=1.2, edgecolor=PLOT_BLUE,
                                  facecolor=PLOT_BLUE if leaf in kd_met else "none",
                                  alpha=0.3 if leaf in kd_met else 1))
    panels[1].text(lower[0], upper[1], str(leaf), ha="left", va="top", fontsize=11,
                   fontweight="bold", color=PLOT_BLUE, zorder=4)
    centre, radius = ball_centres[0, leaf], ball_nodes["radius"][leaf]
    panels[2].add_patch(Circle(centre, radius, linewidth=1.2, edgecolor=PLOT_ORANGE,
                               facecolor=PLOT_ORANGE if leaf in ball_met else "none",
                               alpha=0.3 if leaf in ball_met else 1))
    panels[2].text(centre[0], centre[1] + radius, str(leaf), ha="center", va="bottom",
                   fontsize=11, fontweight="bold", color=PLOT_ORANGE, zorder=4)
for panel in panels[1:]:
    panel.scatter(sample[:, 0], sample[:, 1], s=16, color="lightgrey", zorder=3)
for panel, (name, count) in zip(panels, computed.items()):
    panel.add_patch(Circle(query[0], reach, fill=False, edgecolor="black", linewidth=1.5, zorder=4))
    panel.scatter(*query[0], marker="*", s=160, color="black", zorder=5)
    panel.set_title(f"{name}: {count} distances computed")
    panel.set_xlabel(f"{pair_columns[0]} (prepared)")
    panel.set_aspect("equal")
panels[0].set_ylabel(f"{pair_columns[1]} (prepared)")
plt.show()

**Brute force computed 40 distances, the k-d tree 30 and the ball tree 37, to find a fifth neighbour 0.4279 away.** The circle meets leaves 3, 4 and 5, which hold 30 athletes: the k-d tree computed exactly those and skipped leaf 6, while the ball tree computed the same 30 plus one distance to the centre of each of its 7 balls, which it needs before it can skip a ball.

__Step 24:__ On the course frame, with every prepared column: the distances each method computes per test athlete, with the model's leaf size, `30`.

In [ ]:
per_athlete = {"brute": float(len(X_churn_train_ready))}
for method, tree in [("kd_tree", KDTree(X_churn_train_ready, leaf_size=30)),
                     ("ball_tree", BallTree(X_churn_train_ready, leaf_size=30))]:
    tree.reset_n_calls()
    tree.query(X_churn_test_ready, k=5)
    per_athlete[method] = tree.get_n_calls() / len(X_churn_test_ready)
per_athlete

__Step 25:__ Time each method's fit and prediction on the course frame, and check that the three agree on the test rows.

In [ ]:
timing_rows, method_labels = [], {}
for method in tqdm(["brute", "kd_tree", "ball_tree"], desc="['brute', 'kd tree', 'ball tree']", leave=True, position=0):
    start = time.perf_counter()
    model = KNeighborsClassifier(algorithm=method).fit(X_churn_train_ready, y_churn_train)
    fitted_at = time.perf_counter()
    method_labels[method] = model.predict(X_churn_test_ready)
    timing_rows.append({"algorithm": method, "distances_per_athlete": per_athlete[method],
                        "fit_seconds": fitted_at - start,
                        "predict_seconds": time.perf_counter() - fitted_at})
pd.DataFrame(timing_rows).set_index("algorithm").round(4)

In [ ]:
print("same test predictions as brute force:",
      all(np.array_equal(values, method_labels["brute"]) for values in method_labels.values()))

__Step 26:__ `'auto'`'s choice, read from a fitted default model.

In [ ]:
# _fit_method is the attribute in which a fitted model keeps the method it used.
KNeighborsClassifier().fit(X_churn_train_ready, y_churn_train)._fit_method

<div class="alert alert-block alert-warning">

**On the course frame the trees compute fewer distances and still take longer.** Per test athlete the k-d tree computes 1,571.4 distances and the ball tree 3,169.8, against 3,200, yet both take longer to build and to predict, with the same predictions. In this many columns few boxes or balls are far enough to skip, while brute force computes in large, fast blocks. `'auto'` chose `'brute'`, as scikit-learn does above 15 columns.

</div>

__Step 27:__ Time the methods where a tree can pay, on generated rows: a normal sample from `np.random.RandomState(33)`, so that rows and columns can vary freely. Each time is the median of five repeats of finding five neighbours for 1,000 rows, as the stored rows grow with two columns and as the columns grow with 20,000 stored rows.

In [ ]:
# Generated rows, not course data: a seeded normal sample whose rows and
# columns can be varied freely. fit needs a target; kneighbors does not read it.
generator = np.random.RandomState(random_state)
asked = generator.normal(size=(1000, 32))
generated_rows = []
for rows, columns in tqdm([(3_000, 2), (10_000, 2), (20_000, 2), (100_000, 2), (300_000, 2),
                      (20_000, 4), (20_000, 8), (20_000, 16), (20_000, 32)], desc='[(3000, 2), (10000, 2), (20000, 2), (100000, 2), (300000, 2), (20000, 4), (20000, 8), (20000, 16), (20000, 32)]', leave=True, position=0):
    stored = generator.normal(size=(rows, columns))
    for method in ["brute", "kd_tree", "ball_tree"]:
        model = KNeighborsClassifier(algorithm=method).fit(stored, generator.randint(0, 2, rows))
        seconds = []
        for repeat in range(5):
            start = time.perf_counter()
            model.kneighbors(asked[:, :columns])
            seconds.append(time.perf_counter() - start)
        generated_rows.append({"rows": rows, "columns": columns, "method": method,
                               "milliseconds": 1000 * np.median(seconds)})
generated = pd.DataFrame(generated_rows)

In [ ]:
# Two columns: query time against the stored rows, and brute force's time over the k-d tree's.
by_rows = generated[generated["columns"] == 2].pivot(
    index="rows", columns="method", values="milliseconds")
by_rows["brute / kd_tree"] = by_rows["brute"] / by_rows["kd_tree"]
by_rows.round(2)

In [ ]:
# 20,000 stored rows: query time against the columns, with the same ratio.
by_columns = generated[generated["rows"] == 20_000].pivot(
    index="columns", columns="method", values="milliseconds")
by_columns["brute / kd_tree"] = by_columns["brute"] / by_columns["kd_tree"]
by_columns.round(2)

In [ ]:
colours = {"brute": "#4A4A4A", "kd_tree": PLOT_BLUE, "ball_tree": PLOT_ORANGE}
markers = {"brute": "s", "kd_tree": "o", "ball_tree": "^"}
figure, (left, right) = plt.subplots(1, 2, figsize=(13, 5))
for method in ["brute", "kd_tree", "ball_tree"]:
    for panel, table in [(left, by_rows), (right, by_columns)]:
        panel.plot(table.index, table[method], marker=markers[method], markersize=8,
                   linewidth=2, color=colours[method], label=method)
left.set(xscale="log", yscale="log", xlabel="stored rows, two columns (log scale)",
         ylabel="milliseconds to find five neighbours of 1,000 rows (log scale)",
         title="Query time against the number of stored rows")
right.set(xscale="log", yscale="log", xlabel="columns, 20,000 stored rows (log scale)",
          title="Query time against the number of columns")
left.set_xticks(by_rows.index, labels=[f"{rows:,}" for rows in by_rows.index])
right.set_xticks(by_columns.index, labels=by_columns.index)
for panel in (left, right):
    panel.minorticks_off()
    panel.yaxis.set_major_formatter("{x:g}")
left.legend()
plt.show()

**On two columns the k-d tree is faster than brute force at every size, and at 300,000 stored rows brute force takes more than ten times as long (`brute / kd_tree`).** At 20,000 stored rows the k-d tree is still faster with 4 columns and slower from 8 on, and the ball tree beats brute force only on two columns from 100,000 rows. Timings vary by machine: read the orderings.

# <font color='#E8800A'>The settings of KNN</font> <a class="anchor" id="knn-params"></a>
[Back to TOC](#toc)

Four settings of `KNeighborsClassifier` move here, each alone with the others at their defaults: `n_neighbors`, `weights`, `p` and `algorithm`. Each step prints validation and test F1 side by side and logs its best value; one grid over the combinations makes the choice, after the radius section.

**`n_neighbors`: how many athletes vote.** One neighbour copies the nearest training athlete's outcome; many neighbours average over a wide region. Small `k` is the more complex model, and at `k=1` the training F1 is 1 by construction, each training athlete being its own nearest neighbour.

**Prediction:** training F1 falls from 1 as `k` grows, and validation F1 rises, then falls.

__Step 28:__ Set the values of `n_neighbors` to try: `[1, 3, 5, 9, 15, 21, 31, 41, 61]`. At `5` it is at its default.

In [ ]:
k_values = ...

In the loop, the model is `KNeighborsClassifier(n_neighbors=value)`, and the sweep prints training, validation and test F1 for each value.

In [ ]:
k_rows = []
for value in tqdm(k_values, desc='k values', leave=True, position=0):
    model = ...
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in churn_prepared]
    labels = clone(model).fit(X_churn_train_ready, y_churn_train).predict(X_churn_test_ready)
    row = {"configuration": f"KNeighborsClassifier(n_neighbors={value})", "n_neighbors": value,
           "train": [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"]))
                     for fold, fitted in zip(churn_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], fitted.predict(fold["X_score"]))
                          for fold, fitted in zip(churn_prepared, fold_models)],
           "model": model}
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_churn_test, labels)
    k_rows.append(row)
k_sweep = pd.DataFrame(k_rows)
k_sweep.set_index("n_neighbors")[["train_F1", "cv_F1", "test_F1"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = k_sweep.loc[k_sweep["cv_F1"].idxmax()]
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": best["configuration"],
                   "train": best["train"], "validation": best["validation"],
                   "model": clone(best["model"]).fit(X_churn_train_ready, y_churn_train)},
       X_churn_test_ready, y_churn_test, "one setting changed", metric=f1_score, label="F1")

In [ ]:
plt.figure(figsize=figsize)
plt.plot(k_sweep["n_neighbors"], k_sweep["train_F1"], marker="o", markersize=8, linewidth=2,
         color=PLOT_ORANGE, label="training (ten folds)")
plt.plot(k_sweep["n_neighbors"], k_sweep["cv_F1"], marker="o", markersize=8, linewidth=2,
         color=PLOT_BLUE, label="validation (ten folds)")
plt.axvline(5, color="grey", linestyle=":", linewidth=1.5, label="the default")
plt.xscale("log")
plt.xticks(k_sweep["n_neighbors"], k_sweep["n_neighbors"])
plt.minorticks_off()
plt.xlabel("n_neighbors (log scale)")
plt.ylabel("F1")
plt.title("Training and validation F1 against n_neighbors")
plt.legend()
plt.show()

**Training F1 falls from 1.0000 at `k=1` to 0.8465 at 61, while validation F1 rises to 0.8416 at `k=15` and then stays between 0.8396 and 0.8416.** `KNeighborsClassifier(n_neighbors=15)` joins the log, against the default's 0.8315.

**`weights`: how much each neighbour counts.** `'uniform'` gives each of the five one vote; `'distance'` weights each by one over its distance, so a training athlete, at distance zero from itself, takes all the weight and the training F1 is 1 by construction.

**Prediction:** the training F1 reaches 1, and the validation F1 moves far less.

__Step 29:__ Set the two weightings: `["uniform", "distance"]`; `"uniform"` is the default.

In [ ]:
weight_values = ["uniform", "distance"]

In [ ]:
weight_rows = []
for value in tqdm(weight_values, desc='weight values', leave=True, position=0):
    model = KNeighborsClassifier(weights=value)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in churn_prepared]
    labels = clone(model).fit(X_churn_train_ready, y_churn_train).predict(X_churn_test_ready)
    row = {"configuration": f"KNeighborsClassifier(weights='{value}')", "weights": value,
           "train": [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"]))
                     for fold, fitted in zip(churn_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], fitted.predict(fold["X_score"]))
                          for fold, fitted in zip(churn_prepared, fold_models)],
           "model": model}
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_churn_test, labels)
    weight_rows.append(row)
weight_sweep = pd.DataFrame(weight_rows)
weight_sweep.set_index("weights")[["train_F1", "cv_F1", "test_F1"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = weight_sweep.loc[weight_sweep["cv_F1"].idxmax()]
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": best["configuration"],
                   "train": best["train"], "validation": best["validation"],
                   "model": clone(best["model"]).fit(X_churn_train_ready, y_churn_train)},
       X_churn_test_ready, y_churn_test, "one setting changed", metric=f1_score, label="F1")

**Under `'distance'` the training F1 is 1.0000, as predicted, and the validation F1 0.8305 against 0.8315 under `'uniform'`, which joins the log with the default's scores.**

**`p`: what "near" means.** `p` is the power of the Minkowski distance after Step 4: `p=1` adds the absolute differences, and a larger `p` lets the largest difference decide more.

**Prediction:** the neighbours, and F1, change with `p`.

__Step 30:__ Set the powers to try: `[1, 2, 3]`; `2`, the Euclidean distance, is the default.

In [ ]:
p_values = [1, 2, 3]

In [ ]:
p_rows = []
for value in tqdm(p_values, desc='p values', leave=True, position=0):
    model = KNeighborsClassifier(p=value)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in churn_prepared]
    labels = clone(model).fit(X_churn_train_ready, y_churn_train).predict(X_churn_test_ready)
    row = {"configuration": f"KNeighborsClassifier(p={value})", "p": value,
           "train": [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"]))
                     for fold, fitted in zip(churn_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], fitted.predict(fold["X_score"]))
                          for fold, fitted in zip(churn_prepared, fold_models)],
           "model": model}
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_churn_test, labels)
    p_rows.append(row)
p_sweep = pd.DataFrame(p_rows)
p_sweep.set_index("p")[["train_F1", "cv_F1", "test_F1"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = p_sweep.loc[p_sweep["cv_F1"].idxmax()]
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": best["configuration"],
                   "train": best["train"], "validation": best["validation"],
                   "model": clone(best["model"]).fit(X_churn_train_ready, y_churn_train)},
       X_churn_test_ready, y_churn_test, "one setting changed", metric=f1_score, label="F1")

**`p=1`, the Manhattan distance, scores 0.8372 on the validation folds, against 0.8315 for the Euclidean default and 0.8245 for `p=3`, and joins the log.**

**`algorithm`: how the neighbours are found.** The three methods return the same neighbours.

**Prediction:** only the times move.

__Step 31:__ Set the methods to try: `["auto", "brute", "kd_tree", "ball_tree"]`; `"auto"` is the default.

In [ ]:
algorithm_values = ["auto", "brute", "kd_tree", "ball_tree"]

In [ ]:
algorithm_rows = []
for value in tqdm(algorithm_values, desc='algorithm values', leave=True, position=0):
    model = KNeighborsClassifier(algorithm=value)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in churn_prepared]
    labels = clone(model).fit(X_churn_train_ready, y_churn_train).predict(X_churn_test_ready)
    row = {"configuration": f"KNeighborsClassifier(algorithm='{value}')", "algorithm": value,
           "train": [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"]))
                     for fold, fitted in zip(churn_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], fitted.predict(fold["X_score"]))
                          for fold, fitted in zip(churn_prepared, fold_models)],
           "model": model}
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_churn_test, labels)
    algorithm_rows.append(row)
algorithm_sweep = pd.DataFrame(algorithm_rows)
algorithm_sweep.set_index("algorithm")[["train_F1", "cv_F1", "test_F1"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = algorithm_sweep.loc[algorithm_sweep["cv_F1"].idxmax()]
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": best["configuration"],
                   "train": best["train"], "validation": best["validation"],
                   "model": clone(best["model"]).fit(X_churn_train_ready, y_churn_train)},
       X_churn_test_ready, y_churn_test, "one setting changed", metric=f1_score, label="F1")

__Step 32:__ Time each method on the ten prepared folds: the fits, then the predictions of the validation athletes.

In [ ]:
time_rows = []
for value in tqdm(algorithm_values, desc='algorithm values', leave=True, position=0):
    start = time.perf_counter()
    fold_models = [KNeighborsClassifier(algorithm=value).fit(fold["X_fit"], fold["y_fit"])
                   for fold in churn_prepared]
    fitted_at = time.perf_counter()
    for fold, fitted in zip(churn_prepared, fold_models):
        fitted.predict(fold["X_score"])
    time_rows.append({"algorithm": value, "fit_seconds": fitted_at - start,
                      "predict_seconds": time.perf_counter() - fitted_at})
algorithm_times = pd.DataFrame(time_rows).set_index("algorithm")
algorithm_times.round(4)

**All four rows score 0.8315 on the validation folds and 0.8364 on the test rows: the method changes the time, not the neighbours.** `'auto'` and `'brute'`, the same method here, are the fastest to fit, and `'auto'`, the first of the tied rows, joins the log by the course rule.

# <font color='#E8800A'>Radius neighbours, the alternative</font> <a class="anchor" id="radius"></a>
[Back to TOC](#toc)

`n_neighbors` fixes how many athletes answer; **radius neighbours** fixes how far the answer may come from. Every training athlete within the distance `radius` votes, so an athlete in a crowded region gets many neighbours and one in a sparse region few or none.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.RadiusNeighborsClassifier.html'>sklearn.neighbors.RadiusNeighborsClassifier(radius=1.0, *, weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='minkowski', outlier_label=None, metric_params=None, n_jobs=None)</a>

**Definition:**
Radius neighbours classification: the class held by most stored rows within `radius` of a new row.

**Main Parameters:**
- `radius` (default `1.0`): the largest distance at which a stored row votes.
- `weights` and `p`: as for `KNeighborsClassifier`.
- `outlier_label` (default `None`): the class for a row with no stored row within `radius`; `None` raises an error, `'most_frequent'` gives the most frequent training class.

**Main Attributes (after fitting):**
- `outlier_label_`: the class given to a row with no neighbour.

**Main Methods:**
- `radius_neighbors(X)`: the distances and positions of every stored row within `radius`.
- `predict(X)` and `predict_proba(X)`
</div>

**`radius`: how far a neighbour may be.** Too small, and an athlete in a sparse region has no one to vote; larger than every distance, and every athlete gets the vote of the whole training set.

**Prediction:** F1 rises as the neighbourhoods fill, then falls as they reach unlike athletes.

__Step 33:__ The default radius here: the share of validation athletes with no training athlete within `radius=1.0`, and at five other radii; then the loneliest validation athlete's nearest distance.

In [ ]:
count_rows = []
for radius in tqdm([0.5, 1.0, 1.5, 2.0, 3.0, 4.0], desc='[0.5, 1.0, 1.5, 2.0, 3.0, 4.0]', leave=True, position=0):
    counts = [np.array([len(found) for found in RadiusNeighborsClassifier(radius=radius).fit(
                  fold["X_fit"], fold["y_fit"]).radius_neighbors(fold["X_score"], return_distance=False)])
              for fold in churn_prepared]
    count_rows.append({"radius": radius,
                       "share_with_no_neighbour": np.mean([np.mean(found == 0) for found in counts]),
                       "median_neighbours": np.mean([np.median(found) for found in counts])})
pd.DataFrame(count_rows).set_index("radius").round(4)

In [ ]:
loneliest = max(KNeighborsClassifier(n_neighbors=1).fit(fold["X_fit"], fold["y_fit"]).kneighbors(
    fold["X_score"])[0].max() for fold in tqdm(churn_prepared, desc='churn prepared', leave=True, position=0))
print(f"loneliest validation athlete: {loneliest:.3f} from the nearest training athlete")

**At the default `radius=1.0`, 0.6523 of the validation athletes have no training athlete within reach.** The loneliest is 3.686 from the nearest, and at 4.0 the median athlete has 1,686.5 neighbours: one radius has to serve crowded and sparse athletes alike.

__Step 34:__ Set the values of `radius` to try: `[1.0, 1.5, 2.0, 2.5, 3.0, 4.0]`. At `1.0` it is at its default.

In [ ]:
radius_values = ...

In the loop, the model is `RadiusNeighborsClassifier(radius=value, outlier_label="most_frequent")`: an athlete with no training athlete within the radius gets the most frequent training outcome.

In [ ]:
radius_rows = []
for value in tqdm(radius_values, desc='radius values', leave=True, position=0):
    model = ...
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in churn_prepared]
    labels = clone(model).fit(X_churn_train_ready, y_churn_train).predict(X_churn_test_ready)
    row = {"configuration": f"RadiusNeighborsClassifier(radius={value}, outlier_label='most_frequent')",
           "radius": value,
           "train": [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"]))
                     for fold, fitted in zip(churn_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], fitted.predict(fold["X_score"]))
                          for fold, fitted in zip(churn_prepared, fold_models)],
           "model": model}
    row["cv_F1"], row["test_F1"] = np.mean(row["validation"]), f1_score(y_churn_test, labels)
    radius_rows.append(row)
radius_sweep = pd.DataFrame(radius_rows)
radius_sweep.set_index("radius")[["cv_F1", "test_F1"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = radius_sweep.loc[radius_sweep["cv_F1"].idxmax()]
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": best["configuration"],
                   "train": best["train"], "validation": best["validation"],
                   "model": clone(best["model"]).fit(X_churn_train_ready, y_churn_train)},
       X_churn_test_ready, y_churn_test, "one setting changed", metric=f1_score, label="F1")

__Step 35:__ The share of validation athletes `outlier_label` answers at each radius tried.

In [ ]:
outlier_share = pd.Series(
    [np.mean([np.mean([len(found) == 0 for found in RadiusNeighborsClassifier(radius=value).fit(
        fold["X_fit"], fold["y_fit"]).radius_neighbors(fold["X_score"], return_distance=False)])
              for fold in churn_prepared]) for value in tqdm(radius_values, desc='radius values', leave=True, position=0)],
    index=pd.Index(radius_values, name="radius"), name="cv_outlier_share")
outlier_share.round(4)

**`radius=2.0` scores best, 0.8301 on the validation folds, just below the default KNN's 0.8315.** At 1.0 `outlier_label` answers 0.6523 of the validation athletes and F1 is 0.7530; beyond 2.0 F1 falls again. `RadiusNeighborsClassifier(radius=2.0, outlier_label='most_frequent')` joins the log.

# <font color='#E8800A'>Nested cross-validation</font> <a class="anchor" id="nested"></a>
[Back to TOC](#toc)

The grid fits every combination: 66, of which 54 for `KNeighborsClassifier` (`n_neighbors`, `weights` and `p`) and 12 for `RadiusNeighborsClassifier` (`radius` and `weights`), in one `CachedSearchCV` over a list of two grids whose `estimator` entry names the model.

Nested cross-validation estimates how that whole grid procedure scores on rows that took no part in its choice: the grid runs inside each of the ten outer folds, its inner folds pick the settings from the outer training part, and the outer validation part scores the result. It returns no model, and its mean is read beside the default's validation F1 on the same folds.

__Step 36:__ Define the grid. `algorithm` stays out: its sweep changed the time and not the neighbours. `p` stays out of the radius grid: a radius is a length measured with the distance `p` defines, and the radius values were set under the default, `p=2`.

In [ ]:
# A list of two grids: in each, "estimator" names the model and the entries
# after it set that model's settings.
churn_template = CachedSearchCV(
    PreparedEstimator(FoldPreprocessor(classification_recipe), KNeighborsClassifier()),
    [{"estimator": [KNeighborsClassifier()], "estimator__n_neighbors": k_values,
      "estimator__weights": weight_values, "estimator__p": p_values},
     {"estimator": [RadiusNeighborsClassifier(outlier_label="most_frequent")],
      "estimator__radius": radius_values, "estimator__weights": weight_values}],
    scoring="f1", cv=churn_folds, n_jobs=4,
)

__Step 37:__ Estimate the grid with nested cross-validation.

In [ ]:
churn_nested_fold_results = [cross_validate(churn_template, X_churn_train, y_churn_train, cv=[_outer_split], scoring={'f1': 'f1', 'accuracy': 'accuracy'}, return_train_score=True)
    for _outer_split in tqdm(list(churn_folds.split(X_churn_train, y_churn_train)), total=churn_folds.get_n_splits(X_churn_train, y_churn_train), desc='Nested CV: outer folds', unit='fold', leave=True, position=0)]
churn_nested = {key: np.concatenate([np.asarray(result[key], dtype=object if key == 'estimator' else None) for result in churn_nested_fold_results]) for key in churn_nested_fold_results[0]}
churn_nested_rows = []
for metric in ("f1", "accuracy"):
    validation_values = churn_nested[f"test_{metric}"]
    churn_nested_rows.append({
        "metric": metric,
        "outer_train_mean": churn_nested[f"train_{metric}"].mean(),
        "outer_validation_mean": validation_values.mean(),
        "fold_sd": validation_values.std(ddof=1),
    })
print(pd.DataFrame(churn_nested_rows).round(4).to_string(index=False))

**F1 0.8518 over the ten outer folds, with a fold standard deviation of 0.0107.** The outer training mean, 0.9601, says little, because distance-weighted models score 1 on their own training athletes, and the repeated folds share rows, so their scatter is not an independent-sample standard error.

__Step 38:__ Set the default's validation F1 on the same folds beside the nested mean, and compare their paired fold scores.

In [ ]:
paired_gain = churn_nested["test_f1"] - knn_cv["validation"]
print(f"the default on the same folds: F1 {np.mean(knn_cv['validation']):.4f}")
print(f"nested minus default: mean {paired_gain.mean():+.4f};"
      f" fold range {paired_gain.min():+.4f} to {paired_gain.max():+.4f};"
      f" grid higher in {(paired_gain > 0).sum()} of {len(paired_gain)} folds")

**The nested mean is 0.0203 above the default's 0.8315, and the grid is higher in all ten folds, by 0.0056 to 0.0336.** This is a descriptive comparison of overlapping splits, not a significance test.

# <font color='#E8800A'>The grid search</font> <a class="anchor" id="search"></a>
[Back to TOC](#toc)

The grid now runs once on all the training athletes, and its winner is the model this week deploys.

__Step 39:__ Run the grid on all the training athletes.

In [ ]:
churn_search = clone(churn_template).fit(X_churn_train, y_churn_train)

__Step 40:__ List the five best combinations, each beside the test F1 of its refit.

In [ ]:
results = churn_search.cv_results_
ranking = pd.DataFrame({
    "model": [type(candidate["estimator"]).__name__ for candidate in results["params"]],
    "settings": [", ".join(f"{name.removeprefix('estimator__')}={value}"
                           for name, value in candidate.items() if name != "estimator")
                 for candidate in results["params"]],
    "cv_F1": results["mean_test_score"],
}).nlargest(5, "cv_F1")
ranking["test_F1"] = [
    f1_score(y_churn_test, clone(churn_search.estimator).set_params(
        **clone(results["params"][index], safe=False)).fit(
            X_churn_train, y_churn_train).predict(X_churn_test))
    for index in tqdm(ranking.index, desc='ranking.index', leave=True, position=0)]
print(f"{len(results['params'])} combinations; the five best:")
print(ranking.round(4).to_string())

**The five best are all `KNeighborsClassifier` with `p=1` and distance weighting, differing only in `n_neighbors`, from 15 to 61, at validation F1 0.8508 to 0.8522.** No radius model is among them, since one radius has to serve crowded and sparse athletes alike. The winner, `n_neighbors=41`, scores 0.8615 on the test rows.

__Step 41:__ Log the winner's own score: its validation F1 on the same folds.

In [ ]:
# The winner's own score is the best of all of them on the same folds: its settings
# are fitted on each fold's prepared training part.
winner_model = churn_search.best_estimator_.estimator
fold_models = [clone(winner_model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(churn_prepared, desc='churn prepared', leave=True, position=0)]
winner_churn = {
    "train": [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"]))
              for fold, fitted in zip(churn_prepared, fold_models)],
    "validation": [f1_score(fold["y_score"], fitted.predict(fold["X_score"]))
                   for fold, fitted in zip(churn_prepared, fold_models)]}
# The log names the winner by its settings.
winner_settings = ranking.loc[churn_search.best_index_]
churn_winner_name = f"{winner_settings['model']}({winner_settings['settings']})"
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": churn_winner_name,
                   "train": winner_churn["train"],
                   "validation": winner_churn["validation"],
                   "model": churn_search.best_estimator_},
       X_churn_test, y_churn_test, "winner's own score", metric=f1_score, label="F1")

__Step 42:__ Log the winner's nested estimate, beside the same deployed model's test score.

In [ ]:
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(),
                   "name": f"joint search over {len(churn_search.cv_results_['params'])} combinations",
                   "train": churn_nested["train_f1"],
                   "validation": churn_nested["test_f1"],
                   "model": churn_search.best_estimator_},
       X_churn_test, y_churn_test, "nested estimate", metric=f1_score, label="F1")

**The winner's own validation F1, 0.8522, sits above its nested estimate, 0.8518,** because it was chosen as the best of 66 on the same folds; the board publishes the nested estimate.

# <font color='#E8800A'>The number we stand behind</font> <a class="anchor" id="closing"></a>
[Back to TOC](#toc)

The grid's winner was chosen on the training folds. The closing comparison reports its test score beside the default, and the board publishes that score alongside the nested estimate.

__Step 43:__ Report the winner and the default on the test rows.

In [ ]:
chosen_knn = churn_search.best_estimator_
tuned_f1 = f1_score(y_churn_test, chosen_knn.predict(X_churn_test))
default_f1 = f1_score(y_churn_test, knn_fitted.predict(X_churn_test))

**On the 800 held-out athletes the winner scores 0.8615 and the default 0.8364**, against 0.8518 nested and 0.8315 on the folds. The configuration was chosen using validation scores alone; the test scores shown during the sweeps did not select it.

__Step 44:__ Read three test metrics for the default and the winner: F1, ROC-AUC and balanced accuracy.

In [ ]:
print(f"{'':10s}{'F1':>9s}{'ROC-AUC':>10s}{'balanced acc':>14s}")
for name, model in (("default", knn_fitted), ("tuned", chosen_knn)):
    predicted = model.predict(X_churn_test)
    print(f"{name:10s}{f1_score(y_churn_test, predicted):9.4f}"
          f"{roc_auc_score(y_churn_test, model.predict_proba(X_churn_test)[:, 1]):10.4f}"
          f"{balanced_accuracy_score(y_churn_test, predicted):14.4f}")

__Step 45:__ Take the default's cross-validated F1, on the same folds, as the baseline.

In [ ]:
# The default was measured on these same folds, so it is the baseline.
default_cv = np.mean(knn_cv["validation"])
print(f"default over the outer folds: F1 {default_cv:.4f}")

__Step 46:__ Build the course scoreboard with the results earlier weeks saved and the floor.

In [ ]:
# The board: a result an earlier week saved joins it only if it was scored on the
# same data, recipe, rows and folds; the floor labels every athlete with the
# majority outcome.
results_dir = "../../results"
churn_board = Scoreboard("champions outcome", "F1", context=result_context(
    "../../data/interim/champions.csv", classification_recipe_log,
    y_churn_train.index.tolist(), y_churn_test.index.tolist(),
    outer_folds=outer_folds(churn_folds, X_churn_train, y_churn_train)),
    floor=round(f1_score(y_churn_test, DummyClassifier(strategy="most_frequent").fit(
        X_churn_train, y_churn_train).predict(X_churn_test)), 4))
for week, model, cv, test in results_before(8, "classification", results_dir,
                                             metric=churn_board.metric,
                                             context=churn_board.context):
    churn_board.record(week, model, cv, test)

__Step 47:__ Record this week's result on the board, save it with the fitted winner, and print the board.

In [ ]:
# The row carries two numbers: the nested cross-validated mean first, then the
# winner's held-out score. Its label names the winning model and every setting
# the grid chose for it, from best_params_.
settings = churn_search.best_params_
chosen_label = (type(settings["estimator"]).__name__ + "(" + ", ".join(
    f"{name.removeprefix('estimator__')}={value!r}"
    for name, value in settings.items() if name != "estimator") + ")")
churn_board.record(8, chosen_label,
                   round(churn_nested["test_f1"].mean(), 4), round(tuned_f1, 4))
churn_board.save(
    results_dir, 8, "classification", baseline_cv=round(default_cv, 4),
    baseline_test=round(default_f1, 4),
)

# The fitted winner is saved beside its result, so a later week can reload it if
# it ends the course as the best model. Loading needs preprocessing.py on the path.
joblib.dump(chosen_knn, f"{results_dir}/week_08_classification_model.joblib")

print(churn_board.table())

<div class="alert alert-block alert-success">

**`KNeighborsClassifier(n_neighbors=41, p=1, weights='distance')` joins the board at F1 0.8518 over the ten outer folds and 0.8615 held out**, above the floor's 0.7529 and the logistic regression's 0.8486 and 0.8512, though on the folds by less than the fold standard deviation, 0.0107. The row and the fitted winner are saved in `results/`.

</div>

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. **A prediction is the vote of the nearest stored rows:** six possible probabilities, and an answer even for a probe 64.665 from every training athlete.
2. **The search method changes the time, not the neighbours:** 1,571.4 distances per test athlete against 3,200 for the k-d tree, and still slower here.
3. **One radius cannot serve crowded and sparse athletes alike:** at the default radius 0.6523 of the validation athletes had no neighbour.
4. **Nested cross-validation put the grid at 0.8518, above the default's 0.8315 in all ten folds.**

Exit ticket: on these athletes, with every prepared column, `algorithm="kd_tree"`
returned the same predictions as `algorithm="brute"` and took longer. In one sentence,
why did the tree not save time here?

In [ ]:
# Write your answer below. No Python is required.
#


# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- Cover, T. & Hart, P. (1967). Nearest neighbor pattern classification. *IEEE Transactions on Information Theory*, 13(1), 21-27.
- Bentley, J.L. (1975). Multidimensional binary search trees used for associative searching. *Communications of the ACM*, 18(9), 509-517. (The k-d tree.)
- Omohundro, S.M. (1989). *Five balltree construction algorithms*. Technical Report TR-89-063, International Computer Science Institute, Berkeley. (The ball tree.)
- Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning*, 2nd ed. Springer, § 13.3 (k-nearest neighbours).
- James, G., Witten, D., Hastie, T. & Tibshirani, R. *An Introduction to Statistical Learning*, § 2.2.3.
- scikit-learn User Guide, [Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html) · [`KNeighborsClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) · [`RadiusNeighborsClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.RadiusNeighborsClassifier.html) · [`KDTree`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KDTree.html) · [`BallTree`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.BallTree.html).